# Scaling Laws: fit the line, find the optimum, do the 6ND math

Imagine someone hands you a blank cheque for one GPU cluster and a single question: *if I spend \$10
million training a language model, how good will it be — and should I buy a bigger model or more
data?* A decade ago the only honest answer was "train it and find out." That is a terrifying way to
spend eight figures.

**Scaling laws** are the discovery that changed this. Across many orders of magnitude, a transformer's
test loss falls as a smooth **power law** in model size $N$, data $D$, and compute $C$ — a *straight
line on log-log axes*. Straight lines extrapolate, so a handful of small cheap runs predict the loss of
a giant run you haven't done yet.

This notebook makes the laws **visible from scratch**, in four small pieces. **By the end you'll have
fit a power law and recovered its exponent from "measured" data, found the compute-optimal $(N, D)$
split that minimizes loss for a fixed budget (the famous ~20 tokens/param), worked the $C = 6ND$
calculator on GPT-3 and Chinchilla, and extrapolated the loss to 10× and 100× compute.** Every number
here matches the [Scaling Laws concept page](../03-Scaling-Laws.md) and the [runnable script](scaling_laws.py).

## The problem: you can't afford to guess

A pretraining run has three knobs you control and one number you care about:

- **$N$** — model **parameters** (non-embedding). More capacity to fit patterns.
- **$D$** — **training tokens** seen. More examples to learn from.
- **$C$** — **compute** in FLOPs (what you pay for). It is *not* free to choose: $C \approx 6ND$ ties it to $N$ and $D$.
- **$L$** — **test loss** (cross-entropy, nats/token). The thing you drive down.

If $L$ were a chaotic function of $N$ and $D$ you'd have to try every combination. The miracle is that
it isn't: $L$ is a clean **power law**, so you can *fit* it on small runs and *extrapolate*. We'll prove
that's possible by fitting one ourselves.

## What a power law is, and why loss has a floor

A **power law** means $L - E = A \, N^{-\alpha}$: subtract the irreducible floor $E$ and the *reducible*
loss is a straight line of slope $-\alpha$ on **log-log** axes. Two facts to hold onto:

- The exponent $\alpha$ (small, ~0.05–0.4 for LMs) is the **only** thing that sets how fast you improve —
  it's the slope of that line.
- Loss never reaches zero. It bottoms out at $E$, the **entropy of the data** — the genuinely unpredictable
  part. Scaling buys the *reducible* part only.

The combined form that owns everything below is

$$L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}$$

— floor, plus a **model-size penalty**, plus a **data penalty**. Whichever penalty dominates is your bottleneck.

## How we'll use it: fit, allocate, calculate, extrapolate

Four moves, one per code section below:

1. **Fit.** Train (here: *simulate*) a ladder of small models, measure each loss, fit the log-log line, read off $\alpha$.
2. **Allocate.** For a fixed budget $C$, slide along $D = C/(6N)$ and find the $N^\ast$ that minimizes $L(N, D)$ — the bottom of a **U-shaped** iso-compute curve. That's the compute-optimal split (~20 tokens/param).
3. **Calculate.** Given $N$ and $D$, compute $C = 6ND$ directly — the arithmetic behind every budget.
4. **Extrapolate.** Fit the compute power law $L(C) = E + (C_c/C)^{\alpha_C}$ and predict the loss at 10× and 100× the budget — the move that lets a lab commit to a run it hasn't done.

(The notebook orders them fit-first for teaching; the [companion script](scaling_laws.py) runs the same four as numbered demos `[1]`–`[4]`.)

> **Step 1 — imports and device.** We use `numpy` for the noisy synthetic data and the closed-form fit,
> and `torch` only so the demo is device-agnostic and to print the version. The math is exact arithmetic,
> so the printed numbers are identical on CUDA / MPS / CPU; we pin the trace to **CPU** so this notebook is
> byte-for-byte reproducible on any machine.

In [1]:
import numpy as np
import torch

# Run on the best accelerator available; CPU is the universal fallback.
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
trace_device = "cpu"  # pin the trace so the numbers are reproducible on any machine
print("device:", device, "(trace pinned to", trace_device + ")")
print("torch:", torch.__version__)

device: mps (trace pinned to cpu)
torch: 2.12.0


## Section 1 — the $C = 6ND$ calculator

Start with the compute relation, because it links the budget to $N$ and $D$. **Why the factor 6?** Almost
all transformer compute is in matmuls. Per parameter per token: the **forward** pass does one multiply +
one add = **2** FLOPs; the **backward** pass computes both input-gradients and weight-gradients ≈ **2×** the
forward = **4** FLOPs. Forward + backward = $2 + 4 = 6$ FLOPs per parameter per token, over $D$ tokens:

$$C \approx 6 \, N \, D.$$

> **Step 2 — name the FLOP constants, then define the calculator.** No magic `6` buried in code: we build
> it from the forward (2) and backward (4) counts so the derivation is visible in the names.

In [2]:
FLOPS_PER_PARAM_FWD = 2   # one multiply + one add per weight per token
FLOPS_PER_PARAM_BWD = 4   # backward computes input-grads AND weight-grads ~= 2x forward
FLOPS_PER_PARAM = FLOPS_PER_PARAM_FWD + FLOPS_PER_PARAM_BWD   # = 6, the factor in C = 6ND
FLOPS_PER_SECOND_1PF = 1e15   # 1 petaFLOP/s
SECONDS_PER_DAY = 86_400

def training_flops(n_params, n_tokens):
    """Training compute via C = 6ND."""
    return FLOPS_PER_PARAM * n_params * n_tokens

print("FLOPs per parameter per token (fwd+bwd):", FLOPS_PER_PARAM)

FLOPs per parameter per token (fwd+bwd): 6


> **Step 3 — predict, then compute, for GPT-3 and Chinchilla.** GPT-3 is **175B** params on **300B**
> tokens; Chinchilla is **70B** params on **1.4T** tokens. **Predict before running:** which one has more
> *tokens per parameter*? (Chinchilla — by design: it's the compute-optimal one. GPT-3's ratio is famously
> tiny.) Run it and read the `tokens/param` column.

In [3]:
for name, n, d in [("GPT-3", 175e9, 300e9), ("Chinchilla", 70e9, 1.4e12)]:
    c = training_flops(n, d)
    pf_days = c / (FLOPS_PER_SECOND_1PF * SECONDS_PER_DAY)
    print(f"{name:>11}: N={n:.0e}, D={d:.1e} -> C=6ND={c:.3e} FLOPs "
          f"({pf_days:.0f} PF-days), tokens/param={d/n:.1f}")

# the arithmetic is exact: GPT-3 = 6 * 175e9 * 300e9 = 3.15e23 FLOPs
assert abs(training_flops(175e9, 300e9) - 3.15e23) < 1e18
assert abs(training_flops(70e9, 1.4e12) - 5.88e23) < 1e18
print("\nGPT-3 saw ~1.7 tokens/param; Chinchilla ~20 -> GPT-3 was badly UNDERTRAINED.")

      GPT-3: N=2e+11, D=3.0e+11 -> C=6ND=3.150e+23 FLOPs (3646 PF-days), tokens/param=1.7
 Chinchilla: N=7e+10, D=1.4e+12 -> C=6ND=5.880e+23 FLOPs (6806 PF-days), tokens/param=20.0

GPT-3 saw ~1.7 tokens/param; Chinchilla ~20 -> GPT-3 was badly UNDERTRAINED.


## Section 2 — fit a power law and recover its exponent

This is the heart of the whole topic: *a power law is a straight line on log-log axes, and a straight
line extrapolates.* We'll generate a synthetic "measured" ladder from a **known** law
$L = E + A/N^{\alpha}$ (with a known $\alpha = 0.30$ plus a little measurement noise), then **recover**
$\alpha$ by fitting $\log(L - E)$ against $\log N$. If the fit lands back on 0.30, we've shown the law
is real and fittable — exactly what labs do before a big run.

> **Step 4 — make the synthetic ladder.** Six model sizes spanning $10^4$ to $3\times10^6$ params. We bake
> in the ground-truth exponent so we have something to recover. (`default_rng(0)` makes the noise — and so
> the whole demo — deterministic.)

In [4]:
TRUE_E, TRUE_A, TRUE_ALPHA = 1.55, 18.0, 0.30   # ground-truth law we will try to recover
LADDER_NS = np.array([1e4, 3e4, 1e5, 3e5, 1e6, 3e6])

rng = np.random.default_rng(0)
clean = TRUE_E + TRUE_A / LADDER_NS**TRUE_ALPHA           # the true power law
losses = clean + rng.normal(0, 0.01, LADDER_NS.shape)     # 'measured' = true law + small noise

for n, l in zip(LADDER_NS, losses):
    print(f"N={n:>10.0f}  measured loss={l:.4f}")

N=     10000  measured loss=2.6870
N=     30000  measured loss=2.3655
N=    100000  measured loss=2.1256
N=    300000  measured loss=1.9604
N=   1000000  measured loss=1.8299
N=   3000000  measured loss=1.7588


> **Step 5 — predict, then fit.** **Predict before running:** on log-log axes, plotting $\log(L - E)$ vs
> $\log N$ should give a straight line — what slope do you expect? (About $-0.30$: the slope of a power law
> *is* $-\alpha$.) We fit that slope in closed form (least-squares, no SciPy) and report $\alpha = -\text{slope}$.

In [5]:
def fit_power_law_exponent(n_sizes, measured, irreducible):
    """Recover alpha by a log-log linear fit of the REDUCIBLE loss (L - E) vs N."""
    log_n = np.log(n_sizes)
    log_reducible = np.log(measured - irreducible)        # isolate A/N^alpha before logging
    # least-squares slope of log_reducible = c - alpha * log_n:
    slope = np.polyfit(log_n, log_reducible, 1)[0]
    return -slope                                         # log-log slope is -alpha

recovered = fit_power_law_exponent(LADDER_NS, losses, TRUE_E)
print(f"recovered alpha_N = {recovered:.3f}   (true {TRUE_ALPHA})")
assert abs(recovered - TRUE_ALPHA) < 0.02, "fit should recover the true exponent"
print("The fit recovered the exponent -> the power law is real and extrapolatable.")

recovered alpha_N = 0.299   (true 0.3)
The fit recovered the exponent -> the power law is real and extrapolatable.


## Section 3 — the compute-optimal frontier (the ~20 tokens/param rule)

Now the central question, made concrete. Fix a compute budget $C$. The constraint $6ND = C$ means a bigger
model leaves fewer tokens, and vice versa. **Of all $(N, D)$ pairs that cost exactly $C$, which minimizes
loss?** Slide along the iso-compute line $D = C/(6N)$: loss first *falls* (model too small, underfit) then
*rises* (model too big, starved of data). The minimum in between is the **compute-optimal** point.

We use the Chinchilla parametric fit $L(N,D) = E + A/N^{\alpha} + B/D^{\beta}$ (their fitted constants) to
evaluate the loss at each point.

> **Step 6 — define the parametric loss and the iso-compute sweep.** For a fixed budget, sweep $N$, set
> $D = C/(6N)$ to stay on the budget, and evaluate $L(N, D)$ at each $N$.

In [6]:
# Chinchilla parametric fit (Hoffmann et al. 2022, Table A3):
E, A, B, ALPHA, BETA = 1.69, 406.4, 410.7, 0.34, 0.28

def parametric_loss(n_params, n_tokens):
    """L(N,D) = E + A/N^alpha + B/D^beta : floor + model penalty + data penalty."""
    return E + A / n_params**ALPHA + B / n_tokens**BETA

def iso_compute_curve(budget, n_grid):
    d_grid = budget / (FLOPS_PER_PARAM * n_grid)   # D = C/(6N): stay on the budget
    return d_grid, parametric_loss(n_grid, d_grid)

print("defined parametric loss and the iso-compute sweep")

defined parametric loss and the iso-compute sweep


> **Step 7 — predict the shape, then print the U-curve.** **Predict before running:** as $N$ grows along
> the fixed budget, does loss go *down*, *up*, or *down-then-up*? (Down-then-up — a U. Too-small $N$ underfits;
> too-big $N$ starves on data.) We print a coarse sweep so the U is readable in text, then a fine sweep to
> find the exact minimum.

In [7]:
budget = 1e21   # a fixed compute budget, in FLOPs

# coarse grid -> read the U-shape by eye:
coarse_N = np.logspace(7, 13, 7)
coarse_D, coarse_L = iso_compute_curve(budget, coarse_N)
print(f"budget C = {budget:.0e} FLOPs   (loss is U-shaped along 6ND = C)")
print(f"{'N (params)':>12} | {'D (tokens)':>12} | {'L(N,D)':>8}")
print('-' * 38)
for n, d, l in zip(coarse_N, coarse_D, coarse_L):
    print(f"{n:>12.2e} | {d:>12.2e} | {l:>8.4f}")

# fine grid -> the exact minimum (the compute-optimal N*):
fine_N = np.logspace(7, 13, 200_000)
fine_D, fine_L = iso_compute_curve(budget, fine_N)
i = np.argmin(fine_L)
N_star, D_star, L_star = fine_N[i], fine_D[i], fine_L[i]
print(f"\noptimum: N*={N_star:.2e}, D*={D_star:.2e}, L*={L_star:.4f}, "
      f"tokens/param={D_star/N_star:.0f}")
assert 1e7 < N_star < 1e13, "the optimum is interior (a real minimum, not a grid edge)"

budget C = 1e+21 FLOPs   (loss is U-shaped along 6ND = C)
  N (params) |   D (tokens) |   L(N,D)
--------------------------------------
    1.00e+07 |     1.67e+13 |   3.4657
    1.00e+08 |     1.67e+12 |   2.6198
    1.00e+09 |     1.67e+11 |   2.3400
    1.00e+10 |     1.67e+10 |   2.4160
    1.00e+11 |     1.67e+09 |   2.8389
    1.00e+12 |     1.67e+08 |   3.7722
    1.00e+13 |     1.67e+07 |   5.6085

optimum: N*=1.82e+09, D*=9.14e+10, L*=2.3289, tokens/param=50


> **Step 8 — the canonical 20:1 rule.** The *parametric* optimum above runs a bit above 20 tokens/param
> (its exponents 0.34/0.28 aren't exactly equal). The *equal-exponent* rule — Chinchilla's iso-FLOP
> finding — gives the clean number everyone quotes. Set $D = 20N$ and $C = 6N(20N) = 120 N^2$, solve for $N$.

In [8]:
TOKENS_PER_PARAM = 20
# C = 6 * N * (20N) = 120 N^2  ->  N* = sqrt(C / 120)
N_rule = np.sqrt(budget / (FLOPS_PER_PARAM * TOKENS_PER_PARAM))
D_rule = TOKENS_PER_PARAM * N_rule
print(f"20:1 rule: N*={N_rule:.2e} ({N_rule/1e9:.2f}B), "
      f"D*={D_rule:.2e} ({D_rule/1e9:.0f}B), tokens/param={D_rule/N_rule:.0f}")
# check it lands back on the budget:
assert abs(training_flops(N_rule, D_rule) - budget) / budget < 1e-9
print("6ND check: 6 * N_rule * D_rule == budget  (so this split spends exactly C)")

20:1 rule: N*=2.89e+09 (2.89B), D*=5.77e+10 (58B), tokens/param=20
6ND check: 6 * N_rule * D_rule == budget  (so this split spends exactly C)


## Section 4 — extrapolate the loss to 10× and 100× compute

The payoff. Once the law is fit, you can *predict a run you haven't done*. The loss along the
compute-optimal frontier is itself a power law in compute, $L(C) = E + (C_c/C)^{\alpha_C}$, where
$\alpha_C$ is small (Kaplan's directly-fitted $\approx 0.057$). Because the reducible term is
$(C_c/C)^{\alpha_C}$, **multiplying $C$ by 10 multiplies that term by exactly $10^{-\alpha_C} \approx
0.877$** — a *fixed, small* shrink per decade. That is the brutal economics of the frontier: each
further 10× of compute buys the same small slice of loss, so you must keep paying exponentially more.

> **Step 9 — predict, then extrapolate.** We fix a fitted compute law ($E = 1.69$, $C_c = 3\times10^8$,
> $\alpha_C = 0.057$) and read off the loss at 1×, 10×, 100× a base budget $C_0 = 10^{21}$. **Predict
> before running:** going from 1× to 10×, will the loss drop by a lot or a little? (A little — only the
> *reducible* part shrinks, and only by the factor $10^{-\alpha_C}\approx0.877$.)

In [9]:
EXTRAP_E, EXTRAP_CC, EXTRAP_ALPHA_C = 1.69, 3.0e8, 0.057   # fitted compute power law L(C)
C0 = 1e21                                                  # base budget

def predicted_loss(budget):
    """L(C) = E + (Cc/C)^alpha_C : the loss along the compute-optimal frontier."""
    return EXTRAP_E + (EXTRAP_CC / budget) ** EXTRAP_ALPHA_C

for factor in (1, 10, 100):
    print(f"{factor:4d}x compute -> L = {predicted_loss(factor * C0):.4f}")

# each 10x of compute multiplies the REDUCIBLE term by exactly 10^-alpha_C:
shrink = (predicted_loss(10 * C0) - EXTRAP_E) / (predicted_loss(C0) - EXTRAP_E)
print(f"\n10x shrinks the reducible term by 10^-alpha_C = {10 ** -EXTRAP_ALPHA_C:.4f}")
assert abs(shrink - 10 ** -EXTRAP_ALPHA_C) < 1e-9, "10x should scale reducible loss by 10^-alpha_C"
print("a 10x compute increase buys only ~0.024 nats -> the frontier is brutally capital-intensive.")

   1x compute -> L = 1.8833
  10x compute -> L = 1.8595
 100x compute -> L = 1.8387

10x shrinks the reducible term by 10^-alpha_C = 0.8770
a 10x compute increase buys only ~0.024 nats -> the frontier is brutally capital-intensive.


## Try it yourself

Before you change anything, **predict**: in Section 3 we used budget $C = 10^{21}$. If you raise it to
$C = 10^{24}$ (a thousand times more compute), does the optimal **tokens-per-parameter** ratio stay
roughly the same, or grow? 

Then change `budget` and re-run Steps 7–8. *Hint:* under the **equal-exponent rule**, $N^\ast$ and
$D^\ast$ both grow like $\sqrt{C}$, so their **ratio is constant** — ~20 at every budget. Under the
**parametric** fit (0.34 vs 0.28), the ratio drifts slowly upward. That small disagreement between the two
methods *in the same Chinchilla paper* is real — the field rounds to **~20 tokens/param** as the working rule.

> To do the *real* thing, fit the power law to losses you measure from your own training loop (swap the
> synthetic ladder in Section 2 for your numbers) and extrapolate the line to a budget bigger than anything
> you trained. That extrapolation is exactly how labs commit to nine-figure runs.

## What we saw

- **The power law was real and fittable.** From six noisy "measured" losses, the log-log fit recovered the
  true exponent **$\alpha = 0.30$** (we got 0.299) — a straight line on log-log axes, which is what makes
  extrapolation possible.
- **The compute-optimal split is a U-curve minimum.** For a fixed budget, loss fell then rose as $N$ grew;
  the minimum gave a real interior $N^\ast$, and the equal-exponent rule pinned the canonical **~20 tokens
  per parameter**.
- **$C = 6ND$ is exact arithmetic.** GPT-3 = $3.15 \times 10^{23}$ FLOPs at ~1.7 tokens/param (badly
  undertrained); Chinchilla = $5.88 \times 10^{23}$ at ~20 tokens/param (compute-optimal).
- **Extrapolation buys a fixed, small slice per decade.** Predicting along $L(C) = E + (C_c/C)^{\alpha_C}$,
  each 10× of compute multiplied the reducible loss by $10^{-\alpha_C} \approx 0.877$ — only ~0.024 nats
  from 1× to 10×. That tiny, *predictable* gain is exactly what lets labs commit to nine-figure runs, and
  why the frontier is so capital-intensive.

**What we skipped** (and where to read it): the Lagrangian derivation of why $N^\ast, D^\ast \propto
\sqrt{C}$; *why Kaplan's original recipe was biased toward bigger models*; inference-aware scaling (why
LLaMA over-trains small models on purpose); the data wall; and the emergent-abilities "mirage" debate. All
of that is in the [Scaling Laws concept page](../03-Scaling-Laws.md), with its [curated references](../03-Scaling-Laws.references.md).